# YOLO26 訓練 — PCB 裸板瑕疵偵測（Colab）

在 Colab 上訓練 `yolo26s`，偵測 HRIPCB 資料集的 6 類 PCB 裸板瑕疵：
`missing_hole`、`mouse_bite`、`open_circuit`、`short`、`spur`、`spurious_copper`。

**這個 notebook 要跑兩次**（切換下方 `SPLIT_STRATEGY`）：
1. `"grouped"` — 主要結果，板級分組防洩漏切分（8 板 train / 1 板 val / 1 板 test）
2. `"random"` — 對照組，隨機 80/10/10，與文獻可比但有板子背景洩漏

**預估時間**：yolo26s @ imgsz=640，訓練集數百張圖，150 epochs（patience=30 可能提早停止），T4 上約 1–2 小時，L4/A100 更快。Drive 空間需求 < 1 GB。

**使用前**：
1. `Runtime → Change runtime type → GPU`（Colab Pro 建議選 L4 或以上）
2. 把這個 repo push 到 GitHub（AGPL-3.0 要求原始碼公開，portfolio 專案本來就該公開）,再把下方 `REPO_URL` 換成你自己的網址
3. `Run all`；中途會跳出 Google Drive 授權視窗，同意即可

**test split 政策**：這個 notebook **絕不**在 test split 上驗證或選模型 — test set 留給 Phase 2 在本機只用一次，確保它是真正沒被任何模型選擇/調參決策看過的資料。這裡只用 `val` split。

**斷線怎麼辦**：每個 epoch 的 checkpoint 都直接寫在 Google Drive 上（`last.pt`/`best.pt`），斷線重連後把下方設定改成 `RESUME = True`，再 `Run all` 一次即可從中斷處繼續（前面的 cell 都是冪等的）。

In [ ]:
# ============================================================
# Config — 執行前先確認/修改這裡
# ============================================================

REPO_URL = "https://github.com/YOUR_USERNAME/pcb-defect-detection.git"  # <-- 換成你自己的repo

SPLIT_STRATEGY = "grouped"  # "grouped"（主要）或 "random"（對照）— 每個值各跑一次這個 notebook

RUN_NAME = f"yolo26s_pcb_{SPLIT_STRATEGY}_640"
DRIVE_ROOT = "/content/drive/MyDrive/pcb_defect"
DATA_ROOT = "/content/datasets/pcb"  # 本地磁碟，絕不放 Drive — Drive I/O 會拖垮 dataloader
SEED = 42

RESUME = False  # 斷線重連後：改成 True，再 Run all 一次

assert "YOUR_USERNAME" not in REPO_URL, "請先把 REPO_URL 換成你自己 push 上去的 GitHub repo"
print(f"strategy={SPLIT_STRATEGY}  run_name={RUN_NAME}  resume={RESUME}")

In [ ]:
# 只裝 ultralytics + kagglehub，不動 Colab 預裝的 torch（版本已針對 Colab GPU 優化）
!pip install -q ultralytics==8.4.89 kagglehub

import ultralytics
ultralytics.checks()

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/artifacts", exist_ok=True)
print(f"Drive ready at {DRIVE_ROOT}")

In [ ]:
import os

if not os.path.isdir("/content/repo"):
    !git clone {REPO_URL} /content/repo
else:
    !git -C /content/repo pull

# base dependencies only (kagglehub/pillow/pyyaml) — the [train] extra is NOT
# requested here, so this never touches Colab's own torch install
!pip install -q -e /content/repo

In [ ]:
!python -m pcb_defect.data_prep.prepare --out {DATA_ROOT} --strategy {SPLIT_STRATEGY} --seed {SEED}

In [ ]:
# 視覺+機器雙重檢查：確認資料轉換與切分數字正確，才進入訓練
import json

with open(f"{DATA_ROOT}/conversion_report.json", encoding="utf-8") as f:
    conversion_report = json.load(f)
with open(f"{DATA_ROOT}/split_report.json", encoding="utf-8") as f:
    split_report = json.load(f)

assert conversion_report["images_total"] == 693, conversion_report
assert conversion_report["boxes_total"] == 2953, conversion_report

for split, counts in split_report["image_count_per_split_class"].items():
    missing = [cls for cls, n in counts.items() if n == 0]
    assert not missing, f"{split} split is missing classes: {missing}"

print("OK: 693 images / 2953 boxes, 每個 split 都涵蓋全部 6 類")
print(json.dumps(split_report["images_per_split"], indent=2))
if split_report["board_to_split"]:
    print(json.dumps(split_report["board_to_split"], indent=2, ensure_ascii=False))

## 訓練

`yolo26s`、imgsz=640、150 epochs、patience=30（early stop）。增強參數已針對 PCB 領域調整（理由詳見 repo 的 `plan.md` §1）：色相幾乎不變（銅色/阻焊色本身就是類別訊號）、開啟上下翻轉（PCB 沒有方向性）、關閉 mixup/copy-paste（混疊/貼上的板子不符合物理現實）。

optimizer/lr 留給 YOLO26 的 auto-optimizer 決定（大資料集會自動選 MuSGD 並覆寫手動 `lr0`，這是官方文件記載的行為，不對抗它）。

只用單一 GPU（YOLO26 的多卡 DDP 訓練在官方 repo 有已知 crash — issue #23483；Colab 單卡不受影響）；不使用 `model.tune()`。

In [ ]:
if not RESUME:
    from ultralytics import YOLO

    model = YOLO("yolo26s.pt")
    model.train(
        data=f"{DATA_ROOT}/data.yaml",
        epochs=150,
        patience=30,
        imgsz=640,
        batch=-1,  # auto-batch (~60% VRAM)：自動適應這次配到的 GPU（T4/L4/A100 都適用）
        seed=SEED,
        project=f"{DRIVE_ROOT}/runs",
        name=RUN_NAME,
        exist_ok=True,
        save_period=10,  # 每 10 epoch 留一份快照，Drive 用量可控（~數百 MB）
        # --- 針對 PCB 瑕疵調整的增強參數（理由見 plan.md §1）---
        hsv_h=0.005,
        hsv_s=0.4,
        hsv_v=0.4,
        degrees=10.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        copy_paste=0.0,
        scale=0.5,
        translate=0.1,
    )
else:
    print("RESUME=True — 略過重新訓練，改跑下面的 RESUME cell。")

## 斷線後續訓（只在需要時執行）

`last.pt` 每個 epoch 都直接寫在 Google Drive 上，所以斷線後：
1. 把上面 Config cell 的 `RESUME` 改成 `True`
2. `Run all`（前面的 cell 都是冪等的：重新 mount Drive、重新下載+切分資料 — 因為 seed 固定，切分結果與斷線前完全一致）
3. 下面這格會偵測到 `RESUME=True`，從 `last.pt` 接著訓練

In [ ]:
if RESUME:
    from ultralytics import YOLO

    last_pt = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/last.pt"
    model = YOLO(last_pt)
    model.train(resume=True)  # 重新讀取 last.pt 旁邊的 args.yaml，設定與斷線前一致
    print(f"resumed training from {last_pt}")
else:
    print("RESUME=False — 這格不用做事（只有斷線後才需要）。")

## 驗證（只用 val split）

**這裡只驗證 `val` split，絕不碰 `test` split** — test set 留給 Phase 2 在本機評估一次。

In [ ]:
from pathlib import Path

from IPython.display import Image, display
from ultralytics import YOLO

best_pt = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"
model = YOLO(best_pt)
metrics = model.val(data=f"{DATA_ROOT}/data.yaml", split="val", imgsz=640, plots=True)

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

names = model.names
print(f"\n{'class':<18}{'AP50':>8}{'AP50-95':>10}")
for i, name in names.items():
    print(f"{name:<18}{metrics.box.ap50[i]:>8.4f}{metrics.box.ap[i]:>10.4f}")

val_dir = Path(metrics.save_dir)
print(f"\nval artifacts: {val_dir}")
for fname in ("confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"):
    fpath = val_dir / fname
    if fpath.is_file():
        display(Image(filename=str(fpath)))

In [ ]:
import hashlib
import shutil
from pathlib import Path

run_dir = Path(f"{DRIVE_ROOT}/runs/{RUN_NAME}")
artifact_dir = Path(f"{DRIVE_ROOT}/artifacts/{RUN_NAME}")
artifact_dir.mkdir(parents=True, exist_ok=True)

wanted = [
    "weights/best.pt", "weights/last.pt", "args.yaml", "results.csv",
    "confusion_matrix_normalized.png", "confusion_matrix.png",
    "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png", "results.png",
]
for rel in wanted:
    src = run_dir / rel
    if src.is_file():
        shutil.copy2(src, artifact_dir / src.name)

for pred_img in run_dir.glob("val_batch*_pred.jpg"):
    shutil.copy2(pred_img, artifact_dir / pred_img.name)

best_pt_copy = artifact_dir / "best.pt"
sha256 = hashlib.sha256(best_pt_copy.read_bytes()).hexdigest()
print(f"best.pt SHA-256: {sha256}")

zip_path = shutil.make_archive(str(artifact_dir), "zip", root_dir=artifact_dir)
print(f"packaged: {zip_path}")

print("\nNext（Phase 2，回到本機）：")
print(f"  1. 從 Google Drive 下載 {artifact_dir}/best.pt")
print(f"  2. 存成本機 repo 的 weights/{SPLIT_STRATEGY}/best.pt")
print("  3. 核對上面印出的 SHA-256 與下載後的檔案一致")
if SPLIT_STRATEGY == "grouped":
    print('  4. 接著把上面 SPLIT_STRATEGY 改成 "random"，再 Run all 一次跑對照組')

## （選用）imgsz=1024 對照實驗

`reports/stats.md` 測得瑕疵在 imgsz=640 下換算後中位數約 15px，屬於小物件偵測的困難區間。若想確認提高解析度是否直接帶來更好的小物件偵測（作為 Phase 2 SAHI 實驗的對照基準），可以複製上面的訓練 cell，只把 `imgsz=640` 換成 `imgsz=1024`（`batch=-1` 會自動調整），`RUN_NAME` 也要換一個名字避免覆蓋既有結果。**非必要步驟，Colab 額度有限時可以跳過。**